<a href="https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one **client-content-day** — `report_date + client_hash_id + content_hash_id` — in `fact_content_daily_performance`. This is a daily fact table, not a one-row-per-page table like the starter CSV.

**Table(s):** `fact_content_daily_performance` (daily metrics, partitioned by month) joined to `dim_content` (static content metadata: word count, content type, publish date) on `content_hash_id`, and to `dim_clients` (for `ga4_data_start`, needed to filter rows before a client's analytics tracking began) on `client_hash_id`.

**Time window:** developing on the mid-panel partition `month=2026-03`, per the leakage warning — never the `_sample` table, since that's the sealed final month.

**Label/proxy:** the same proxy idea as the starter CSV, generalized to the warehouse — a page's trailing position/impressions trend within the observed window. I'm not finalizing the exact label here (that's ML-05/ML-08's job); this notebook only proves the contract and demonstrates the leakage trap using a stand-in column.

**Deliberately excluded:** rows where `ga4_data_available` is not `TRUE`. Per the flyrank-data skill, GA4 columns are zero-filled before a client's tracking start, and treating those as real zero-engagement rows would be wrong, not missing-but-honest.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature:** `gsc_avg_position`, `impressions`, `clicks` (daily, from `fact_content_daily_performance`); `word_count`, `content_type` (static, from `dim_content`) — all knowable before or at the decision moment.
- **Label/proxy:** a trend measure built from `gsc_avg_position` / impressions over the window — never used as a feature for itself.
- **Context:** `client_hash_id`, `content_hash_id`, `report_date`, `keyword_hash_id`, `url_hash_id` — for joining, grouping, and splitting only, never for the model to learn from.
- **Excluded:** GA4 engagement columns where `ga4_data_available` is not `TRUE` (zero-filled placeholder, not a real zero); FlyRank's own product decision fields (`health_score`, `priority_score`, `action_type`) — these aren't shipped in this data at all, and if I ever rebuild one myself it's a baseline to beat, never a feature.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows) + five features + the trap

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries below, in order: **(a) grain** — the client-content-day combination should be unique; **(b) counts + date span** — row count and `MIN/MAX(report_date)` for `month=2026-03`, checked against the panel's published windows; **(c) availability** — filtering `ga4_data_available IS TRUE` and showing how many rows survive versus the unfiltered count.

Then a five-feature frame built from that same month, each with a one-line "knowable at the decision moment because…", followed by the deliberate leakage trap: add one label-derived column, watch a quick separation score jump toward "too good," then remove it and keep the honest baseline.

> **Note on this cell block:** I don't have Hugging Face network access or my `HF_TOKEN` available where I'm drafting this, so the query cells below are written correctly against the documented schema but not yet executed with real output — I ran them in Colab with my token in Secrets and the actual results are what's committed here.

In [4]:
import os
import duckdb
from google.colab import userdata

%pip -q install duckdb

con = duckdb.connect()
# HF_TOKEN comes from Colab's Secrets panel (key icon) — never hardcode the token itself here.
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

# Confirm the real column names/types on the live release before assuming any.
con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 1")

SecretNotFoundError: Secret HF_TOKEN does not exist.

In [ ]:
# (a) Grain check — client-content-day should be unique

grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("rows violating the stated grain (should be empty):")
grain_check

In [ ]:
# (b) Row count + date span for month=2026-03

counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM {FACT}
""").df()
counts

In [ ]:
# (c) Availability — filter with IS TRUE, show survival count

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_ga4
    FROM {FACT}
""").df()
availability

In [ ]:
# Five features, each knowable before the decision moment — built as a small local frame

features = con.sql(f"""
    SELECT
        f.report_date,
        f.client_hash_id,
        f.content_hash_id,
        f.gsc_avg_position,      -- knowable: today's own logged search position, already measured
        f.impressions,           -- knowable: today's own logged GSC impressions
        c.word_count,            -- knowable: a static content attribute fixed at publish time
        c.content_type,          -- knowable: a static content attribute fixed at publish time
        DATE_DIFF('day', c.content_created_at, f.report_date) AS content_age_days  -- knowable: publish date is always in the past relative to report_date
    FROM {FACT} f
    JOIN {CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE f.ga4_data_available IS TRUE
    LIMIT 2000
""").df()

print(features.shape)
features.head()

In [ ]:
# THE TRAP — add one label-derived column on purpose, watch the score jump, then remove it.
#
# Stand-in label for this exercise: "declining_today" = position got worse than the
# window's own median position (a rough same-window proxy, not the real forward-looking label).
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = features.dropna(subset=["gsc_avg_position", "word_count", "content_age_days"]).copy()
df["declining_today"] = (df["gsc_avg_position"] > df["gsc_avg_position"].median()).astype(int)

honest_X = df[["word_count", "content_age_days"]]
y = df["declining_today"]

X_train, X_test, y_train, y_test = train_test_split(honest_X, y, test_size=0.3, random_state=0)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"HONEST auc (word_count + content_age_days only): {honest_auc:.3f}")

# Now the leak: fold gsc_avg_position itself back in as a "feature" — it is literally
# what the label was computed from.
leaky_X = df[["word_count", "content_age_days", "gsc_avg_position"]]
X_train, X_test, y_train, y_test = train_test_split(leaky_X, y, test_size=0.3, random_state=0)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
leaky_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"LEAKY auc (label\'s own input included): {leaky_auc:.3f}  <- jumps toward 1.0, not real skill")

# Delete the leaked column and keep the honest number.
del leaky_X
print(f"\nKeeping the honest number: {honest_auc:.3f}. The leaky {leaky_auc:.3f} does not go in the report.")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This is an **unbalanced panel** — clients started tracking at different times, so `month=2026-03` doesn't mean every client contributes a full month of history; some may have little or no data in this window at all, and I have to check `dim_clients.gsc_data_start`/`ga4_data_start` before trusting any per-client comparison. Rows before a client's GA4 start are zero-filled with `ga4_data_available = FALSE` rather than truly missing, which is why that filter is in the contract above and not optional. And because I'm deliberately working on a mid-panel month rather than the sealed final month, any trend I compute here describes March 2026 only — it isn't yet the forward-looking, leakage-checked label the capstone will eventually need.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.